# AI Dustbin — Teacher model (DINOv2 + linear, CLIP baseline)
Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

Before running: in the Files panel, make folder **`/content/zips`** and drag in `dataset_C_v2.zip` and `dataset_D_v2_balanced.zip`.
Outputs go to `/content/teacher_out/`; the last cell downloads them as a zip (Colab deletes files when the session ends).

In [ ]:
!nvidia-smi -L
!pip -q install open_clip_torch

In [ ]:
# ---- CONFIG ----
# Before running: in the Files panel (left), create folder /content/zips and drag in
# dataset_C_v2.zip and dataset_D_v2_balanced.zip
import os, glob, zipfile
OUT  = '/content/teacher_out'
DATA = '/content/data'
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
zips = sorted(glob.glob('/content/zips/*.zip'))
print('Zips found:', [os.path.basename(z) for z in zips])
assert len(zips) == 2, 'Put dataset_C_v2.zip and dataset_D_v2_balanced.zip in /content/zips'
for z in zips:
    dest = f'{DATA}/{os.path.splitext(os.path.basename(z))[0]}'
    if not os.path.exists(dest):
        zipfile.ZipFile(z).extractall(dest)
DATASETS = {os.path.basename(d): d for d in sorted(glob.glob(f'{DATA}/*'))}
print('Datasets:', list(DATASETS))

## 1. Index images
Infers **split** (train/val/test), **label** (dry/wet) and **source** (roboflow/realwaste) from folder names.
Check the printed counts. If the zip has a manifest CSV with these columns, load that instead.
Mask files (dataset D) are skipped.

In [ ]:
import pandas as pd
IMG_EXT = {'.jpg','.jpeg','.png','.bmp','.webp'}
SPLIT_ALIASES = {'train':'train','val':'val','valid':'val','validation':'val','test':'test'}

def index_dataset(root):
    rows = []
    for p in glob.glob(f'{root}/**/*', recursive=True):
        if os.path.splitext(p)[1].lower() not in IMG_EXT: continue
        rel = os.path.relpath(p, root).lower()
        dirs = rel.split(os.sep)[:-1]
        if 'mask' in rel: continue
        split = next((SPLIT_ALIASES[d] for d in dirs if d in SPLIT_ALIASES), None)
        label = next((l for d in dirs for l in ('dry','wet') if l in d), None)
        source = next((s for s in ('roboflow','realwaste') if s in rel), 'unknown')
        rows.append(dict(path=p, split=split, label=label, source=source))
    df = pd.DataFrame(rows)
    df['y'] = (df.label == 'wet').astype(int)      # 0 = dry, 1 = wet (matches dry_prob, wet_prob)
    return df

INDEX = {}
for name, root in DATASETS.items():
    df = index_dataset(root)
    print(f'\n=== {name}: {len(df)} images')
    print(df.groupby(['split','label','source']).size().unstack(fill_value=0))
    bad = df[df.split.isna() | df.label.isna()]
    if len(bad): print('!! could not parse', len(bad), 'files, e.g.', bad.path.iloc[0])
    INDEX[name] = df.dropna(subset=['split','label']).reset_index(drop=True)

# ---- source labels from manifest.csv (match on file name without extension) ----
for name, root in DATASETS.items():
    m = glob.glob(f'{root}/**/manifest.csv', recursive=True)
    if not m: print('no manifest for', name); continue
    man = pd.read_csv(m[0])
    print(name, 'manifest columns:', list(man.columns))
    fcol = next(c for c in man.columns if any(k in c.lower() for k in ('file','path','name')))
    man['stem'] = man[fcol].astype(str).map(lambda s: os.path.splitext(os.path.basename(s))[0])
    df = INDEX[name]
    df['stem'] = df.path.map(lambda p: os.path.splitext(os.path.basename(p))[0])
    df['source'] = df.stem.map(dict(zip(man.stem, man['source']))).fillna('unknown')
    print(df.groupby(['split','source']).size().unstack(fill_value=0))

## 2. Feature extraction helpers

In [ ]:
import torch, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
dev = 'cuda'
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
# Same input as the student: RGB, squashed to 224x224, ImageNet normalisation
base_tf = T.Compose([T.Resize((224,224)), T.ToTensor(), T.Normalize(MEAN, STD)])
aug_tf  = T.Compose([T.RandomResizedCrop(224, scale=(0.7,1.0), ratio=(0.8,1.25)),
                     T.RandomHorizontalFlip(), T.ColorJitter(0.2,0.2,0.2),
                     T.ToTensor(), T.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, paths, tf): self.p, self.tf = list(paths), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB'))

@torch.no_grad()
def extract(fn, paths, tf, bs=64):
    dl = DataLoader(ImgDS(paths, tf), batch_size=bs, num_workers=2)
    out = []
    for x in dl:
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(fn(x.to(dev)).float().cpu())
    return torch.cat(out).numpy()

def cached_features(tag, ds_name, split, fn, tf, n_aug=0):
    """Extracts once, saves to Drive, reloads on rerun."""
    path = f'{OUT}/feats_{tag}_{ds_name}_{split}.npz'
    if os.path.exists(path):
        d = np.load(path); return d['X'], d['y']
    df = INDEX[ds_name]; df = df[df.split == split]
    Xs, ys = [extract(fn, df.path, tf)], [df.y.values]
    for k in range(n_aug):                      # extra augmented views (train only)
        Xs.append(extract(fn, df.path, aug_tf)); ys.append(df.y.values)
    X, y = np.concatenate(Xs), np.concatenate(ys)
    np.savez(path, X=X, y=y); return X, y

## 3. Metrics: accuracy per source, confusion matrix, temperature scaling, ECE

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, log_loss
from scipy.optimize import minimize_scalar

def sigmoid(z): return 1/(1+np.exp(-z))

def fit_temperature(logits, y):
    nll = lambda logT: log_loss(y, np.clip(sigmoid(logits/np.exp(logT)),1e-7,1-1e-7))
    return float(np.exp(minimize_scalar(nll, bounds=(-3,3), method='bounded').x))

def ece(p_wet, y, bins=15):
    conf = np.maximum(p_wet, 1-p_wet); pred = (p_wet >= .5).astype(int)
    e, edges = 0.0, np.linspace(.5, 1, bins+1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m]==y[m]).mean() - conf[m].mean())
    return e

def report(name, p_wet, y, sources):
    pred = (p_wet >= .5).astype(int)
    r = {'model': name, 'acc': (pred==y).mean(), 'ece': ece(p_wet, y)}
    for s in np.unique(sources): r[f'acc_{s}'] = (pred[sources==s]==y[sources==s]).mean()
    cm = confusion_matrix(y, pred, labels=[0,1])
    r['cm [[dd,dw],[wd,ww]]'] = cm.tolist()
    return r

## 4. CLIP zero-shot baseline (no training)

In [ ]:
import open_clip
clip_model, _, clip_pre = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model = clip_model.to(dev).eval()
tok = open_clip.get_tokenizer('ViT-B-32')
PROMPTS = {
  0: ['a photo of dry waste', 'a photo of recyclable waste', 'a photo of plastic, paper, metal or glass trash',
      'a photo of a plastic bottle or cardboard'],
  1: ['a photo of wet waste', 'a photo of food waste', 'a photo of organic kitchen waste',
      'a photo of vegetable peels and leftover food'],
}
with torch.no_grad():
    T_emb = []
    for c in (0,1):
        e = clip_model.encode_text(tok(PROMPTS[c]).to(dev)).float()
        e = e / e.norm(dim=-1, keepdim=True); T_emb.append(e.mean(0))
    T_emb = torch.stack(T_emb); T_emb = T_emb / T_emb.norm(dim=-1, keepdim=True)

clip_fn = lambda x: clip_model.encode_image(x)
RESULTS = []
for ds in INDEX:
    X, y = cached_features('clip', ds, 'test', clip_fn, clip_pre)
    Xn = torch.tensor(X); Xn = Xn / Xn.norm(dim=-1, keepdim=True)
    logits = 100 * Xn @ T_emb.cpu().T
    p_wet = logits.softmax(-1)[:,1].numpy()
    src = INDEX[ds].query('split=="test"').source.values
    RESULTS.append({'dataset': ds, **report('CLIP zero-shot', p_wet, y, src)})
pd.DataFrame(RESULTS)

## 5. Teacher: DINOv2 ViT-B/14 features + logistic regression
Features are extracted once (train gets 2 extra augmented views). Refitting the classifier takes seconds,
which is what the feedback loop needs. Regularisation C is chosen on **val**; test is used once.

In [ ]:
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').to(dev).eval()
dino_fn = lambda x: dino(x)          # CLS embedding, 768-d

def train_linear_teacher(Xtr, ytr, Xva, yva, tag):
    best = None
    for C in [0.01, 0.03, 0.1, 0.3, 1.0]:
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        clf.fit(Xtr, ytr)
        l = log_loss(yva, clf.predict_proba(Xva)[:,1])
        if best is None or l < best[0]: best = (l, C, clf)
    _, C, clf = best
    Tval = fit_temperature(clf.decision_function(Xva), yva)
    print(f'{tag}: best C={C}, temperature={Tval:.3f}')
    return clf, Tval

import joblib
TEACHERS = {}
for ds in INDEX:
    Xtr, ytr = cached_features('dino', ds, 'train', dino_fn, base_tf, n_aug=2)
    Xva, yva = cached_features('dino', ds, 'val',   dino_fn, base_tf)
    Xte, yte = cached_features('dino', ds, 'test',  dino_fn, base_tf)
    clf, Tval = train_linear_teacher(Xtr, ytr, Xva, yva, ds)
    TEACHERS[ds] = (clf, Tval)
    joblib.dump({'clf': clf, 'temperature': Tval, 'backbone': 'dinov2_vitb14'}, f'{OUT}/teacher_dino_{ds}.joblib')
    src = INDEX[ds].query('split=="test"').source.values
    z = clf.decision_function(Xte)
    RESULTS.append({'dataset': ds, **report('DINOv2+linear (uncalibrated)', sigmoid(z), yte, src)})
    RESULTS.append({'dataset': ds, **report('DINOv2+linear (calibrated)', sigmoid(z/Tval), yte, src), 'T': Tval})

## 6. Bonus baseline: CLIP features + linear (same recipe)

In [ ]:
for ds in INDEX:
    Xtr, ytr = cached_features('clip', ds, 'train', clip_fn, clip_pre)
    Xva, yva = cached_features('clip', ds, 'val',   clip_fn, clip_pre)
    Xte, yte = cached_features('clip', ds, 'test',  clip_fn, clip_pre)
    clf, Tval = train_linear_teacher(Xtr, ytr, Xva, yva, 'clip-'+ds)
    src = INDEX[ds].query('split=="test"').source.values
    RESULTS.append({'dataset': ds, **report('CLIP+linear (calibrated)', sigmoid(clf.decision_function(Xte)/Tval), yte, src), 'T': Tval})

res = pd.DataFrame(RESULTS)
res.to_csv(f'{OUT}/teacher_results.csv', index=False)
pd.set_option('display.width', 200); res

## 7. Export soft labels for distillation
One CSV per dataset: every image with the teacher's calibrated `teacher_wet_prob` and raw `teacher_logit`.
Distillation uses these (train/val) with the student. Uses clean (non-augmented) features.

In [ ]:
for ds, (clf, Tval) in TEACHERS.items():
    parts = []
    for split in ['train','val','test']:
        df = INDEX[ds][INDEX[ds].split == split].copy()
        X = extract(dino_fn, df.path, base_tf) if split == 'train' else cached_features('dino', ds, split, dino_fn, base_tf)[0]
        z = clf.decision_function(X)
        df['teacher_logit'] = z; df['teacher_wet_prob'] = sigmoid(z/Tval)
        parts.append(df)
    out = pd.concat(parts)[['path','split','label','source','teacher_logit','teacher_wet_prob']]
    out['path'] = out.path.str.replace(DATA + '/', '', regex=False)
    out.to_csv(f'{OUT}/soft_labels_{ds}.csv', index=False)
    print(ds, 'saved', len(out), 'rows')

## 8. Teacher wrapper for the bin's crops (for the feedback loop later)
Same input as `classify_image()`: one BGR uint8 crop. Returns the same dict shape as the student.

In [ ]:
import time
class TeacherClassifier:
    def __init__(self, joblib_path):
        d = joblib.load(joblib_path); self.clf, self.T = d['clf'], d['temperature']
        self.backbone = torch.hub.load('facebookresearch/dinov2', d['backbone']).to(dev).eval()
    @torch.no_grad()
    def classify_image(self, crop_bgr):
        t0 = time.time()
        x = base_tf(Image.fromarray(crop_bgr[:, :, ::-1].copy())).unsqueeze(0).to(dev)
        f = self.backbone(x).float().cpu().numpy()
        p_wet = float(sigmoid(self.clf.decision_function(f)[0] / self.T))
        label = 'wet waste' if p_wet >= .5 else 'dry waste'
        return {'label': label, 'confidence': max(p_wet, 1-p_wet), 'dry_prob': 1-p_wet,
                'wet_prob': p_wet, 'inference_ms': (time.time()-t0)*1000}

    def refit(self, X_old, y_old, X_new, y_new, X_val, y_val):
        """Feedback round: refit on old data + new human labels (avoids forgetting)."""
        self.clf, self.T = train_linear_teacher(np.concatenate([X_old, X_new]),
                                                np.concatenate([y_old, y_new]), X_val, y_val, 'refit')

# quick smoke test on one test image
import cv2
ds0 = list(INDEX)[0]
t = TeacherClassifier(f'{OUT}/teacher_dino_{ds0}.joblib')
row = INDEX[ds0].query('split=="test"').iloc[0]
print(row.label, t.classify_image(cv2.imread(row.path)))

## 9. Download all outputs (do this before closing Colab)

In [ ]:
!cd /content && zip -qr teacher_out.zip teacher_out
from google.colab import files
files.download('/content/teacher_out.zip')